# Comp Lab 2  -  Protein Structure Prediction and the Limits of Confidence
**BIO 462/594 Molecular Biology | Thursday, November 5 | OSC OnDemand (Pitzer, GPU) | Teams of 2**

Each team predicts the 3D structure of ONE protein with **ESMFold**, then does what a careful scientist does with any prediction: reads the
**confidence scores** critically.

By the end of the session your team can:
1. Run a structure-prediction model and save its output (PDB file).
2. Interpret **pLDDT**  -  per-residue confidence  -  and know its color convention.
3. Interpret the **PAE matrix**  -  per-pair confidence  -  and use it to find domains and flexible linkers.
4. Compare your prediction with the **AlphaFold Database** model of the same protein and judge where the two agree.
5. Say, with reasons, where a confidence score should change how you use a structure  -  and where "low confidence" is actually the *correct* answer.

## How this notebook works
- Set your team number in the first code cell, then run top to bottom with **Shift+Enter**.
- Cells marked **Your turn** ask for a short written answer (double-click a text cell to type) or a small code change.
- The prediction itself is one line of code. The science is in the next 45 minutes of interpretation.

In [ ]:
TEAM = 1   # <-- your team number (1-8). Change this one number, then run everything.

import pandas as pd, numpy as np, matplotlib.pyplot as plt, torch, time
plt.rcParams['font.family'] = ['Liberation Sans', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

menu = pd.read_csv('data/protein_menu.csv')
row = menu[menu['team'] == TEAM].iloc[0]
SEQ = row['sequence']
print(f"Team {TEAM}: {row['protein']}  ({row['uniprot']}, {len(SEQ)} aa)")
print(f"Confidence story you are testing: {row['confidence_story']}")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for gpu_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_index)
        gpu_name = torch.cuda.get_device_name(gpu_index)
        print(
            f'GPU {gpu_index}: {gpu_name}, '
            f'{free_bytes / 1024**3:.1f} GiB free / '
            f'{total_bytes / 1024**3:.1f} GiB total'
        )
else:
    print('CUDA is unavailable. ESMFold will run on CPU and may be very slow.')

---
## Part 1  -  Run ESMFold (15 min)

ESMFold (Lin et al., Science 2022) pairs a large protein language model (ESM-2) with a folding module. Unlike AlphaFold2, it **does not
search a sequence database at prediction time**  -  the language model has already absorbed evolutionary patterns during training. That makes
it ~60x faster, and it is why we can run it live in class.

Loading the model takes a minute (the weights are ~3 GB). The prediction itself takes seconds on a GPU.

In [ ]:
import esm
model = esm.pretrained.esmfold_v1()
model = model.eval().to(device)
print('model ready')

In [ ]:
t0 = time.time()
output = model.infer(SEQ)
prediction_time = time.time() - t0
print(f'prediction took {prediction_time:.1f} s for {len(SEQ)} residues')

In [ ]:
# save the predicted structure as a PDB file (B-factors = per-atom pLDDT)
pdb_string = model.output_to_pdb(output)[0]
with open(f'team{TEAM}_esmfold.pdb', 'w') as f:
    f.write(pdb_string)
print('saved team%d_esmfold.pdb' % TEAM)

---
## Part 2  -  Read the pLDDT (15 min)

**pLDDT** is a per-residue confidence score from 0 to 100, predicted by the model itself. The community color convention (used by the
AlphaFold Database):

| pLDDT | color | meaning |
|---|---|---|
| > 90 | blue | very high accuracy  -  backbone likely correct to ~1 Å |
| 70–90 | teal/cyan | confident  -  backbone mostly right, side chains approximate |
| 50–70 | yellow | low  -  do not trust the exact geometry |
| < 50 | orange | very low  -  probably disordered, or the model has no idea |

**The key habit:** a predicted *structure* is a claim; pLDDT is the model telling you how much it believes its own claim, residue by residue.

In [ ]:
def plddt_from_pdb(pdb_text, chain=None):
    # per-residue pLDDT = B-factor of each residue's CA atom
    out = {}
    for line in pdb_text.splitlines():
        if line.startswith(('ATOM', 'HETATM')) and line[12:16].strip() == 'CA':
            ch, resi, b = line[21], int(line[22:26]), float(line[60:66])
            if chain and ch != chain:
                continue
            out[resi] = b
    return np.array([out[i] for i in sorted(out)])

plddt = plddt_from_pdb(pdb_string)          # one score per residue (CA atom)
residues = np.arange(1, len(plddt) + 1)

fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(residues, plddt, lw=0.8, color='#0279EE')
for y, c, lab in [(90, '#0279EE', '90'), (70, '#00B0C7', '70'), (50, '#FFC800', '50')]:
    ax.axhline(y, color=c, lw=0.8, ls='--')
    ax.text(len(plddt) + 2, y, lab, color=c, va='center', fontsize=9)
ax.set_xlabel('residue number')
ax.set_ylabel('pLDDT')
ax.set_xlim(0, len(plddt) * 1.06)
ax.set_ylim(0, 100)
fig.tight_layout()
fig.savefig(f'team{TEAM}_plddt.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'mean pLDDT: {plddt.mean():.1f} | residues below 50: {(plddt < 50).sum()} ({100*(plddt<50).sum()/len(plddt):.0f}%)')

**Your turn 1 (written answer).** Double-click this cell and type:
1. Which regions of your protein have pLDDT < 70? Give residue ranges.
2. Does that match the confidence story your protein was assigned? (Yes / partially / no  -  and how?)

In [ ]:
# visualize the structure colored by pLDDT (B-factor column)
import py3Dmol
view = py3Dmol.view(width=650, height=480)
view.addModel(pdb_string, 'pdb')
view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'rwb', 'min': 30, 'max': 100}}})
view.zoomTo()
view.show()

Red = low confidence, blue = high confidence in this rendering. Compare with the pLDDT plot above  -  same information, different view.
**Screenshot this for your report.**

---
## Part 3  -  Read the PAE matrix (15 min)

pLDDT says how confident the model is about each residue's *local* geometry. It says nothing about whether two well-folded domains are
oriented correctly relative to each other. That is the job of the **Predicted Aligned Error (PAE)**: for every pair of residues (i, j), the
expected position error (in Å) if you align on residue i and place residue j.

**How to read it:** With the reversed viridis scale used here, bright yellow means low PAE and more confident relative positioning; dark
purple means high PAE and less certain positioning. Low-PAE blocks along the diagonal indicate well-determined domains. Darker off-diagonal
regions indicate uncertainty in how parts are oriented relative to one another, often because of a flexible linker. A mostly low-PAE matrix,
like the lysozyme example, is consistent with a well-defined fold.

In [ ]:
pae = output['predicted_aligned_error'][0].cpu().numpy()   # L x L matrix, Angstroms
fig, ax = plt.subplots(figsize=(5.6, 4.6))
im = ax.imshow(pae, cmap='viridis_r', vmin=0, vmax=min(30, np.percentile(pae, 99)))
fig.colorbar(im, ax=ax, label='PAE (Å)', shrink=0.85)
ax.set_xlabel('aligned residue')
ax.set_ylabel('scored residue')
ax.set_title(f'Team {TEAM}: {row["protein"]}')
fig.tight_layout()
fig.savefig(f'team{TEAM}_pae.png', dpi=150, bbox_inches='tight')
plt.show()

**Your turn 2 (written answer).** Double-click and type:
1. How many dark diagonal blocks do you see, and what does each correspond to in the sequence?
2. Are any two dark blocks connected by a light region? If yes, what does that say about the linker between them?
3. If your protein is one rigid domain, say so  -  that is a valid answer, and the PAE should be dark everywhere.

---
## Part 4  -  Compare with the AlphaFold Database (15 min)

The AlphaFold Database (AFDB) contains an AlphaFold2 prediction for your protein, computed by DeepMind with a different architecture and an
MSA-based pipeline. **Two independent-ish models of the same protein agreeing is evidence; disagreeing is a question.**

In [ ]:
import requests

afdb_id = row['afdb']
if afdb_id.startswith('AF-'):
    url = f'https://alphafold.ebi.ac.uk/files/{afdb_id}-model_v6.pdb'
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    afdb_pdb = r.text
    open(f'team{TEAM}_afdb.pdb', 'w').write(afdb_pdb)
    print('downloaded', afdb_id)
else:
    print('No AFDB entry for this protein - compare to the experimental structure 1QYS instead:')
    r = requests.get('https://files.rcsb.org/download/1QYS.pdb', timeout=60)
    afdb_pdb = r.text
    open(f'team{TEAM}_afdb.pdb', 'w').write(afdb_pdb)
    print('downloaded experimental structure 1QYS')

In [ ]:
af_plddt = plddt_from_pdb(afdb_pdb)
if TEAM == 5:   # RBD: slice the same residues out of the full spike model
    af_plddt = af_plddt[318:541]

n = min(len(plddt), len(af_plddt))
fig, ax = plt.subplots(figsize=(11, 3.2))
ax.plot(residues[:n], plddt[:n], lw=1, color='#0279EE', label='ESMFold (today)')
ax.plot(np.arange(1, n + 1), af_plddt[:n], lw=1, color='#FF9400', alpha=0.85, label='AlphaFold DB')
ax.axhline(70, color='gray', lw=0.8, ls='--')
ax.set_xlabel('residue number'); ax.set_ylabel('pLDDT')
ax.legend(loc='lower left')
fig.tight_layout()
fig.savefig(f'team{TEAM}_plddt_compare.png', dpi=150, bbox_inches='tight')
plt.show()

**Your turn 3 (written answer).** Double-click and type:
1. Where do the two models agree on confidence? Where do they disagree?
2. For your protein: which model's confidence profile makes more biological sense, given what you know about how the protein works?
3. If both models show low confidence in the same region, what are the two possible explanations, and what experiment would tell them apart?

---
## Part 5  -  Share-out and report (10 min)

**Share-out (5 min).** One minute per team: your protein, one sentence on the pLDDT profile, one sentence on the PAE, and whether the
confidence story we assigned was right. Listen for: lysozyme (control), calmodulin (domain PAE), p53 (disordered termini), rhodopsin (TM
bundle), RBD (domain-in-context), Top7 (no homologs), alpha-synuclein (disordered everywhere), histone H3 (needs its partners).

**Structure Report  -  due Thursday, Nov 12 (30% of the portfolio).** Team report, 2–3 pages. Full instructions and rubric in the report
handout. In short: your confidence analysis, the AFDB comparison, and a critical paragraph on what a user of your structure should and
should not do with it.

In [ ]:
# summary for your report
summary = dict(team=TEAM, protein=row['protein'], length=len(SEQ),
               device=device, prediction_seconds=round(prediction_time, 1),
               mean_plddt=round(float(plddt.mean()), 1),
               pct_below_50=round(100 * float((plddt < 50).mean()), 1),
               mean_pae=round(float(pae.mean()), 2))
summary